<a href="https://colab.research.google.com/github/temraire117/NLP/blob/main/NLP_06%EB%B0%B0%ED%8F%AC%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6주차 실습: 텍스트 표현 방법

**주제:나이브 베이즈·로지스틱 회귀 기반 문서 분류**

## 학습목표
1. 지도학습 텍스트 분류의 학습·검증·테스트 흐름을 설명할 수 있다
2. 나이브 베이즈와 로지스틱 회귀의 기본 가정과 출력 차이를 이해할 수 있다
3. TF-IDF 특징으로 문서 분류 모델을 학습·예측할 수 있다
4. 정확도·정밀도·재현율·F1-score·혼동행렬을 해석할 수 있다

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

In [ ]:
# @title 라이브러리 import
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 재현성을 위한 전역 시드 고정
random.seed(42)
np.random.seed(42)

In [ ]:
# @title 데이터셋 생성
# 총 188개(긍정 90+4, 부정 90+4) 정도의 샘플이 생성되며, label 컬럼은 1(긍정)/0(부정)으로 구성
# value_counts()로 클래스 비율을 확인해 두면 이후 stratify 옵션의 필요성을 설명할 때 자연스럽게 연결

# 1) 긍정/부정 핵심 표현
positive_cores = [
    "정말 만족스러워요", "품질이 아주 좋아요", "가격 대비 훌륭해요",
    "디자인이 마음에 들어요", "배송이 빨라서 좋았어요", "사용하기 편리해요",
    "생각보다 훨씬 좋네요", "재구매 의사 있어요", "친절한 서비스였어요",
    "튼튼하고 오래 쓸 것 같아요", "색감이 예뻐서 좋아요", "선물용으로도 훌륭해요",
    "포장이 꼼꼼해서 좋았어요", "기대 이상이에요", "가성비가 최고예요"
]

negative_cores = [
    "정말 불만족스러워요", "품질이 너무 별로예요", "가격이 아깝네요",
    "디자인이 촌스러워요", "배송이 너무 늦었어요", "사용하기 불편해요",
    "생각보다 훨씬 안 좋네요", "다시는 안 살 것 같아요", "불친절한 서비스였어요",
    "금방 고장 났어요", "색감이 사진과 달라요", "선물용으로는 부적합해요",
    "포장이 엉망이었어요", "기대에 못 미쳐요", "가격 대비 최악이에요"
]

subjects = [
    "이 제품은", "이번에 구매한 상품은", "받아보니", "사용해보니",
    "이 브랜드는", "이 옷은", "이 가전은", "이 화장품은", "이 신발은", "이 가방은"
]

# 2) 템플릿 조합으로 문장 생성
def generate_sentences(cores, subjects, label, n):
    sentences = []
    for _ in range(n):
        subj = random.choice(subjects)
        core = random.choice(cores)
        sentences.append((f"{subj} {core}.", label))
    return sentences

positive_samples = generate_sentences(positive_cores, subjects, 1, 90)  # 1 = 긍정
negative_samples = generate_sentences(negative_cores, subjects, 0, 90)  # 0 = 부정

# 3) 오류 분석용 까다로운 사례 (부정어, 반어법, 짧은 문장 등) - 소량 섞기
tricky_cases = [
    ("나쁘지 않아요.", 1),          # 부정어가 있지만 실제 긍정
    ("안 좋은 건 아니에요.", 1),     # 이중 부정 → 긍정
    ("정말 최고네요, 두 번 다시 안 살 거예요.", 0),  # 반어법 → 실제 부정
    ("그냥 그래요.", 0),            # 애매한 중립성 짧은 문장 → 부정으로 라벨
    ("괜찮아요.", 1),               # 짧은 문장, 긍정
    ("별로.", 0),                   # 매우 짧은 문장
    ("가격은 싼데 품질은 그럭저럭이에요.", 0),  # 혼합 감정, 부정 쪽으로 라벨
    ("비싸긴 한데 품질은 정말 좋아요.", 1),      # 혼합 감정, 긍정 쪽으로 라벨
]

# 4) 전체 데이터셋 결합 및 셔플
all_samples = positive_samples + negative_samples + tricky_cases
random.shuffle(all_samples)

df = pd.DataFrame(all_samples, columns=["text", "label"])
print(f"전체 샘플 수: {len(df)}개")
print(df["label"].value_counts())
df.head(10)

전체 샘플 수: 188개
label
1    94
0    94
Name: count, dtype: int64


,text,label
0,이 제품은 선물용으로도 훌륭해요.,1
1,이 브랜드는 재구매 의사 있어요.,1
2,받아보니 친절한 서비스였어요.,1
3,받아보니 친절한 서비스였어요.,1
4,이 제품은 금방 고장 났어요.,0
5,사용해보니 품질이 너무 별로예요.,0
6,이 신발은 가격이 아깝네요.,0
7,사용해보니 가격 대비 훌륭해요.,1
8,이 옷은 정말 불만족스러워요.,0
9,받아보니 배송이 너무 늦었어요.,0


In [ ]:
# @title 슬라이드 12. 데이터 분할 (train/test) — 시드 고정


texts = df["text"].values
labels = df["label"].values

# test_size=0.2, random_state=42로 시드 고정 → 재현성 확보(두 모델(NB, LR)을 "동일한 조건"에서 비교하기 위한 전제 조건)
# stratify=labels → 긍정/부정 비율을 train/test에서 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"학습 데이터: {len(X_train)}개, 테스트 데이터: {len(X_test)}개")
print(f"학습 데이터 라벨 비율: {np.bincount(y_train)}")
print(f"테스트 데이터 라벨 비율: {np.bincount(y_test)}")

학습 데이터: 150개, 테스트 데이터: 38개
학습 데이터 라벨 비율: [75 75]
테스트 데이터 라벨 비율: [19 19]


In [ ]:
# @title 슬라이드 13. Naive Bayes 파이프라인 학습
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.naive_bayes import MultinomialNB

# Pipeline을 사용하면 TF-IDF 벡터화와 분류기 학습을 하나의 객체로 묶을 수 있어,
# 이후 predict()를 호출할 때 원문 텍스트를 그대로 넣어도 내부적으로 벡터화까지 자동으로 처리

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
nb_pipeline.fit(X_train, y_train)
print("Naive Bayes 학습 완료")

Naive Bayes 학습 완료


In [ ]:
# @title 슬라이드 14. 예측 결과 확인 및 오류 샘플 분석


# 예측
nb_preds = nb_pipeline.predict(X_test)####
nb_accuracy = accuracy_score(y_test, nb_preds)
print(f"Naive Bayes 정확도: {nb_accuracy:.3f}")

# 실제 라벨 vs 예측 라벨 비교 표
result_df = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "예측 라벨": nb_preds
})
result_df["일치 여부"] = np.where(result_df["실제 라벨"] == result_df["예측 라벨"], "v", "x")

display(result_df.head(10))

# 오분류 샘플만 추출
errors_df = result_df[result_df["일치 여부"] == "x"]###
print(f"\n오분류 샘플 수: {len(errors_df)}개")
display(errors_df[["문장", "실제 라벨", "예측 라벨"]])

Naive Bayes 정확도: 0.895


,문장,실제 라벨,예측 라벨,일치 여부
0,이 신발은 친절한 서비스였어요.,1,1,v
1,이 신발은 기대에 못 미쳐요.,0,0,v
2,사용해보니 품질이 너무 별로예요.,0,0,v
3,이 제품은 금방 고장 났어요.,0,0,v
4,사용해보니 기대 이상이에요.,1,1,v
5,사용해보니 금방 고장 났어요.,0,0,v
6,이 브랜드는 생각보다 훨씬 안 좋네요.,0,1,x
7,이 브랜드는 기대 이상이에요.,1,1,v
8,사용해보니 재구매 의사 있어요.,1,1,v
9,이 옷은 가격 대비 훌륭해요.,1,1,v



오분류 샘플 수: 4개


,문장,실제 라벨,예측 라벨
6,이 브랜드는 생각보다 훨씬 안 좋네요.,0,1
12,괜찮아요.,1,0
20,이 가전은 생각보다 훨씬 안 좋네요.,0,1
22,가격은 싼데 품질은 그럭저럭이에요.,0,1


In [ ]:
# 라벨을 숫자(0/1)로만 보면 어떤 클래스가 긍정·부정인지 헷갈릴 수 있으므로, 발표나 보고서에는 아래처럼 라벨명을 매핑해 출력하는 것이 읽기 편함.
label_map = {1: "긍정", 0: "부정"}
errors_df_display = errors_df.copy()
errors_df_display["실제 라벨"] = errors_df_display["실제 라벨"].map(label_map)
errors_df_display["예측 라벨"] = errors_df_display["예측 라벨"].map(label_map)
errors_df_display[["문장", "실제 라벨", "예측 라벨"]]


In [ ]:
# @title 슬라이드 15. Logistic Regression 학습 (동일 분할·동일 특징 조건)

# 두 모델 모두 정확히 같은 X_train, X_test, 같은 TfidfVectorizer() 기본 설정을 사용했기 때문에,
# 성능 차이가 나온다면 그 원인을 "분류기 알고리즘 자체의 차이"로 좁혀서 설명 가능함.
# 만약 벡터화 파라미터(ngram_range, min_df 등)까지 다르게 주면 비교의 공정성이 깨지므로,
# 두 파이프라인의 TfidfVectorizer() 설정은 반드시 동일하게 맞춰야함.

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression 학습 완료")

lr_preds = lr_pipeline.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)
print(f"Logistic Regression 정확도: {lr_accuracy:.3f}")

Logistic Regression 학습 완료
Logistic Regression 정확도: 0.895


In [ ]:
# @title 두 모델 성능 한눈에 비교
# 두 모델의 결과를 표와 혼동행렬로 함께 확인

print("=== Naive Bayes ===")
print(classification_report(y_test, nb_preds, target_names=["부정", "긍정"]))
print("혼동행렬:\n", confusion_matrix(y_test, nb_preds))

print("\n=== Logistic Regression ===")
print(classification_report(y_test, lr_preds, target_names=["부정", "긍정"]))
print("혼동행렬:\n", confusion_matrix(y_test, lr_preds))

comparison_df = pd.DataFrame({
    "모델": ["Naive Bayes", "Logistic Regression"],
    "정확도": [nb_accuracy, lr_accuracy]
})
comparison_df

=== Naive Bayes ===
              precision    recall  f1-score   support

          부정       0.94      0.84      0.89        19
          긍정       0.86      0.95      0.90        19

    accuracy                           0.89        38
   macro avg       0.90      0.89      0.89        38
weighted avg       0.90      0.89      0.89        38

혼동행렬:
 [[16  3]
 [ 1 18]]

=== Logistic Regression ===
              precision    recall  f1-score   support

          부정       0.94      0.84      0.89        19
          긍정       0.86      0.95      0.90        19

    accuracy                           0.89        38
   macro avg       0.90      0.89      0.89        38
weighted avg       0.90      0.89      0.89        38

혼동행렬:
 [[16  3]
 [ 1 18]]


,모델,정확도
0,Naive Bayes,0.894737
1,Logistic Regression,0.894737


## 실습

앞서 만든 데이터셋과 파이프라인을 기반으로, 실습 과제 3가지

(①분할 기준·시드값 기록, ②모델 성능 비교표, ③오분류 사례 분석)를 순서대로 처리하는 완성 코드

In [ ]:
# @title 라이브러리 임포트 및 시드 고정

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# @title 데이터셋 생성

positive_cores = [
    "정말 만족스러워요", "품질이 아주 좋아요", "가격 대비 훌륭해요",
    "디자인이 마음에 들어요", "배송이 빨라서 좋았어요", "사용하기 편리해요",
    "생각보다 훨씬 좋네요", "재구매 의사 있어요", "친절한 서비스였어요",
    "튼튼하고 오래 쓸 것 같아요", "색감이 예뻐서 좋아요", "선물용으로도 훌륭해요",
    "포장이 꼼꼼해서 좋았어요", "기대 이상이에요", "가성비가 최고예요"
]

negative_cores = [
    "정말 불만족스러워요", "품질이 너무 별로예요", "가격이 아깝네요",
    "디자인이 촌스러워요", "배송이 너무 늦었어요", "사용하기 불편해요",
    "생각보다 훨씬 안 좋네요", "다시는 안 살 것 같아요", "불친절한 서비스였어요",
    "금방 고장 났어요", "색감이 사진과 달라요", "선물용으로는 부적합해요",
    "포장이 엉망이었어요", "기대에 못 미쳐요", "가격 대비 최악이에요"
]

subjects = [
    "이 제품은", "이번에 구매한 상품은", "받아보니", "사용해보니",
    "이 브랜드는", "이 옷은", "이 가전은", "이 화장품은", "이 신발은", "이 가방은"
]

def generate_sentences(cores, subjects, label, n):
    sentences = []
    for _ in range(n):
        subj = random.choice(subjects)
        core = random.choice(cores)
        sentences.append((f"{subj} {core}.", label))
    return sentences

positive_samples = generate_sentences(positive_cores, subjects, 1, 90)
negative_samples = generate_sentences(negative_cores, subjects, 0, 90)

# 오류 분석용 까다로운 사례 (부정어, 반어법, 짧은 문장, 혼합 감정 등)
tricky_cases = [
    ("나쁘지 않아요.", 1),
    ("안 좋은 건 아니에요.", 1),
    ("정말 최고네요, 두 번 다시 안 살 거예요.", 0),
    ("그냥 그래요.", 0),
    ("괜찮아요.", 1),
    ("별로.", 0),
    ("가격은 싼데 품질은 그럭저럭이에요.", 0),
    ("비싸긴 한데 품질은 정말 좋아요.", 1),
]

all_samples = positive_samples + negative_samples + tricky_cases
random.shuffle(all_samples)

df = pd.DataFrame(all_samples, columns=["text", "label"])
print(f"전체 샘플 수: {len(df)}개")
print(df["label"].value_counts())

전체 샘플 수: 188개
label
1    94
0    94
Name: count, dtype: int64


In [ ]:
# @title 데이터 분할 기준 및 시드값 기록

texts = df["text"].values
labels = df["label"].values

SPLIT_STRATEGY = "train/test (8:2)"   # validation 없이 train/test만 사용
TEST_SIZE = 0.2
RANDOM_STATE = SEED
STRATIFY = True

# TODO: 학습/테스트 데이터 분할 및 시드 고정
X_train, X_test, y_train, y_test = train_test_split(
texts, labels, test_size=0.2, random_state=42, stratify=labels



)

# 재현성을 위한 분할 기준 기록
split_record = pd.DataFrame({
    "항목": ["분할 방식", "test_size", "random_state", "stratify 적용 여부",
             "학습 샘플 수", "테스트 샘플 수", "학습 라벨 비율(0/1)", "테스트 라벨 비율(0/1)"],
    "값": [
        SPLIT_STRATEGY, TEST_SIZE, RANDOM_STATE, STRATIFY,
        len(X_train), len(X_test),
        str(np.bincount(y_train)), str(np.bincount(y_test))
    ]
})

print("=== 데이터 분할 기준 기록 ===")
print(split_record.to_string(index=False))

=== 데이터 분할 기준 기록 ===
            항목                값
         분할 방식 train/test (8:2)
     test_size              0.2
  random_state               42
stratify 적용 여부             True
       학습 샘플 수              150
      테스트 샘플 수               38
 학습 라벨 비율(0/1)          [75 75]
테스트 라벨 비율(0/1)          [19 19]


In [16]:
# @title 모델 성능 비교표 (Naive Bayes vs. Logistic Regression)

# TODO: Naive Bayes 파이프라인 학습

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])
nb_pipeline.fit(X_train, y_train)



# TODO: Logistic Regression 파이프라인 학습 (동일한 분할, 동일한 TF-IDF 조건)

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

lr_preds = lr_pipeline.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)
print(f"Logistic Regression 정확도: {lr_accuracy:.3f}")



#  TODO: 지표 계산 함수 (긍정=1 클래스 기준 precision/recall/f1)
def get_metrics(y_true, y_pred, model_name):
    return {
        "모델": model_name,

        "정확도": accuracy_score(y_true, y_pred),

        "정밀도": precision_score(y_true, y_pred, pos_label=1,zero_division=0),

        "재현율": recall_score(y_true, y_pred, pos_label=1, zero_division=0),

        "F1-score": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

metrics_nb = get_metrics(y_test, nb_preds, "Naive Bayes")
metrics_lr = get_metrics(y_test, lr_preds, "Logistic Regression")

comparison_df = pd.DataFrame([metrics_nb, metrics_lr])
comparison_df = comparison_df.round(3)

print("=== 모델 성능 비교표 (동일 test set 기준) ===")
print(comparison_df.to_string(index=False))

# 세부 classification report도 함께 출력 (0=부정, 1=긍정)
print("\n=== Naive Bayes classification report ===")
print(classification_report(y_test, nb_preds, target_names=["부정", "긍정"]))

print("=== Logistic Regression classification report ===")
print(classification_report(y_test, lr_preds, target_names=["부정", "긍정"]))

# 혼동행렬
print("\nNaive Bayes 혼동행렬:\n", confusion_matrix(y_test, nb_preds))
print("Logistic Regression 혼동행렬:\n", confusion_matrix(y_test, lr_preds))

Logistic Regression 정확도: 0.895
=== 모델 성능 비교표 (동일 test set 기준) ===
                 모델   정확도   정밀도   재현율  F1-score
        Naive Bayes 0.895 0.857 0.947       0.9
Logistic Regression 0.895 0.857 0.947       0.9

=== Naive Bayes classification report ===
              precision    recall  f1-score   support

          부정       0.94      0.84      0.89        19
          긍정       0.86      0.95      0.90        19

    accuracy                           0.89        38
   macro avg       0.90      0.89      0.89        38
weighted avg       0.90      0.89      0.89        38

=== Logistic Regression classification report ===
              precision    recall  f1-score   support

          부정       0.94      0.84      0.89        19
          긍정       0.86      0.95      0.90        19

    accuracy                           0.89        38
   macro avg       0.90      0.89      0.89        38
weighted avg       0.90      0.89      0.89        38


Naive Bayes 혼동행렬:
 [[16  3]
 [ 1 18]]
Logi

In [17]:
# @title 오분류 사례 분석 (5건 + 유형 분류 + 개선 방안)

# Logistic Regression 기준으로 오분류 샘플 확인 (필요시 nb_preds로 바꿔서 비교 가능)
result_df = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "NB 예측": nb_preds,
    "LR 예측": lr_preds
})

label_map = {1: "긍정", 0: "부정"}
for col in ["실제 라벨", "NB 예측", "LR 예측"]:
    result_df[col + "_명"] = result_df[col].map(label_map)

# TODO 두 모델 중 하나라도 틀린 샘플만 추출
errors_df = result_df[
    (result_df["실제 라벨"] != result_df["NB 예측"]) |
    (result_df["실제 라벨"] != result_df["LR 예측"])
].reset_index(drop=True)

print(f"오분류(둘 중 하나라도 틀린) 샘플 수: {len(errors_df)}개\n")
print(errors_df[["문장", "실제 라벨_명", "NB 예측_명", "LR 예측_명"]])

# 상위 5건 선택 (샘플이 5건 미만이면 있는 만큼만)
top5_errors = errors_df.head(5).copy()

# 유형 분류 및 개선 방안을 사람이 직접 채우는 표 (강의 실습용 템플릿)
top5_errors["오류 유형"] = ""       # 예: 부정어, 반어법, 짧은 문장, 라벨링 오류, 혼합감정 등
top5_errors["원인 해석"] = ""       # 예: "부정어(안~않다) 패턴을 TF-IDF가 단어 단위로만 인식해 반영 못함"
top5_errors["개선 방안"] = ""       # 예: "n-gram 추가(ngram_range=(1,2))로 부정어+단어 조합 학습"

print("\n=== 오분류 사례 분석 템플릿 (5건) ===")
print(top5_errors[["문장", "실제 라벨_명", "NB 예측_명", "LR 예측_명",
                    "오류 유형", "원인 해석", "개선 방안"]].to_string(index=False))

오분류(둘 중 하나라도 틀린) 샘플 수: 4개

                      문장 실제 라벨_명 NB 예측_명 LR 예측_명
0  이 브랜드는 생각보다 훨씬 안 좋네요.      부정      긍정      긍정
1                  괜찮아요.      긍정      부정      부정
2   이 가전은 생각보다 훨씬 안 좋네요.      부정      긍정      긍정
3    가격은 싼데 품질은 그럭저럭이에요.      부정      긍정      긍정

=== 오분류 사례 분석 템플릿 (5건) ===
                   문장 실제 라벨_명 NB 예측_명 LR 예측_명 오류 유형 원인 해석 개선 방안
이 브랜드는 생각보다 훨씬 안 좋네요.      부정      긍정      긍정                  
                괜찮아요.      긍정      부정      부정                  
 이 가전은 생각보다 훨씬 안 좋네요.      부정      긍정      긍정                  
  가격은 싼데 품질은 그럭저럭이에요.      부정      긍정      긍정                  


**형성평가 (직접 답을 적어보세요)**

```
Q1. 정확도가 90%여도 긍정/부정 한쪽을 거의 맞히지 못할 수 있는 상황은 언제일까요?
답: 데이터가 불균형할 경우. 극단적인 예시로 실제 부정이 90%고 전체를 부정으로 예측했을 경우를 생각해볼 수 있다.

Q2. 정밀도와 재현율 중, 스팸 필터와 의료 문서 분류에서 각각 더 우선해야 할 지표는 무엇인가요?
답: 스팸 필터에서는 정상 메일을 스팸으로 잘못 분류하는 피해를 줄이기 위해 정밀도를 우선, 의료 문서 분류에서는 중요 문서가 많으므로 재현율을 우선. 다만 실제 우선순위는 오분류로 발생하는 피해에 따라 달라질 수 있다.

Q3. 위에서 만든 comparison_df를 보고, 두 모델의 성능 차이가 전처리(TF-IDF)가 아니라
    분류기 알고리즘 자체에서 왔다고 말할 수 있는 이유는 무엇인가요?
답: 두 모델이 동일한 학습·테스트 데이터 분할과 동일한 TF-IDF 설정을 사용하고, 분류기 알고리즘만 다르게 구성했기 때문.
```